# 01 · Трекинг II: слабые детекции и разные ошибки метрик
К лекции 8. Девять стендов: от отдельных метрик к покадровым решениям трекера. Каждый опыт: прогноз → изменение параметра → измерение → объяснение.

Синтетические сцены изолируют механизмы. Стенды MOTA, IDF1 и HOTA вызывают **официальный TrackEval** закрепленной версии; исходники и лицензия скачиваются в временный кэш с проверкой SHA. Нужны numpy, scipy, matplotlib, ipywidgets и доступ к GitHub при первом запуске. После заполнения кэша сеть не нужна.

Статический и движущийся BYTE-подобные стенды раскрывают приоритет стадий, рождения и lifecycle. Это не полный ByteTrack с YOLOX/Kalman и его инженерными деталями.

Краткая теория находится перед стендом, вопросы — сразу под ним. Перед изменением параметров прочитайте вопросы и запишите прогноз.

In [ ]:
# Запускайте первой. В Colab сохраняем штатные PyTorch/CUDA и ipywidgets.
import importlib.util, subprocess, sys
if sys.version_info < (3, 10):
    raise RuntimeError('Нужен Python 3.10 или новее.')
_packages = {'numpy': 'numpy>=1.26,<3', 'matplotlib': 'matplotlib>=3.8,<4', 'scipy': 'scipy>=1.11,<2', 'ipywidgets': 'ipywidgets>=7.7,<9'}
_missing = [spec for module, spec in _packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
    raise RuntimeError('Пакеты установлены. Перезапустите среду выполнения и выполните ноутбук сверху.')
import numpy, matplotlib
if not (1 <= int(numpy.__version__.split('.')[0]) < 3):
    raise RuntimeError('Нужен NumPy 1.x или 2.x; используйте отдельное окружение.')
if int(matplotlib.__version__.split('.')[0]) != 3:
    raise RuntimeError('Нужен Matplotlib 3.x; используйте отдельное окружение.')
_versions = {'python': sys.version.split()[0], 'numpy': numpy.__version__,
             'matplotlib': matplotlib.__version__}
if 'scipy' in _packages:
    import scipy
    if int(scipy.__version__.split('.')[0]) != 1:
        raise RuntimeError('Нужен SciPy 1.x; используйте отдельное окружение.')
    _versions['scipy'] = scipy.__version__
if 'cv2' in _packages:
    import cv2
    _versions['opencv'] = cv2.__version__
    _required_features = ()
    _absent = [name for name in _required_features if not hasattr(cv2, name)]
    if _absent:
        raise RuntimeError('В OpenCV нет ' + ', '.join(_absent) +
                           '. Установите один вариант OpenCV 4.12.0.88 в отдельной среде и перезапустите ядро.')
if 'ipywidgets' in _packages:
    import ipywidgets
    _versions['ipywidgets'] = ipywidgets.__version__
    if int(ipywidgets.__version__.split('.')[0]) not in (7, 8):
        raise RuntimeError('Нужен ipywidgets 7.x или 8.x; в Colab 2026.07 оставьте штатный 7.7.1.')
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
print('Окружение:', _versions)
print('Вычисления этой лабораторной: CPU; GPU не требуется.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.optimize import linear_sum_assignment
from pathlib import Path
import time,json,csv,os,hashlib
plt.rcParams.update({'figure.dpi':110,'font.size':10})

In [ ]:
# Official source files, pinned revision. License is downloaded alongside code.
import urllib.request,tempfile,types,sys,importlib
TRACKEVAL_REV='12c8791b303e0a0b50f753af204249e622d0281a'
TRACKEVAL_HASHES={'LICENSE': 'f7e548c5729e464d878dd4abf691e323bbea028a756d08395d88b83e43dd2d86', 'trackeval/metrics/hota.py': 'c582255c3d36bdb49c3ceeb2a5bd912cee009190dfcfe465b4173d03aaa97d6a', 'trackeval/metrics/clear.py': 'ab3963eac401073fe1b2957c734af1e531368f42fca49627c14422350b2744f0', 'trackeval/metrics/identity.py': 'e76b1bdcbf5b193662431596a984cf70a5be32e903eb5f2a3d9921b4762f93ba', 'trackeval/metrics/_base_metric.py': 'dac8b87bf3801269aa280b1cf91184f483de4f4e5c158bd07a4b1733b1a5722b', 'trackeval/_timing.py': '62e60638efc675bccc35c7da65158f0f19ce31f332121efe7cc748bd92ad2b30', 'trackeval/utils.py': 'c082ecfbb5061619642b82a034454022806f8e870f774e54dbfbdacaaf097132'}
cache=Path(tempfile.gettempdir())/'applied-ai-trackeval'/TRACKEVAL_REV
root=Path(os.environ.get('COURSE_TRACKEVAL_PATH',cache))
for relative,expected in TRACKEVAL_HASHES.items():
 target=root/relative
 if not target.exists():
  target.parent.mkdir(parents=True,exist_ok=True)
  url=f'https://raw.githubusercontent.com/JonathonLuiten/TrackEval/{TRACKEVAL_REV}/{relative}'
  with urllib.request.urlopen(url,timeout=60) as response:target.write_bytes(response.read())
 assert hashlib.sha256(target.read_bytes()).hexdigest()==expected,'TrackEval source checksum mismatch: '+relative
# Old revision uses removed NumPy builtin aliases. No metric formulas are changed.
if 'float' not in np.__dict__:np.float=float
if 'int' not in np.__dict__:np.int=int
# Load only official metric modules, without unrelated dataset adapters.
namespace='_course_trackeval_pinned'
for name,path in [(namespace,root/'trackeval'),(namespace+'.metrics',root/'trackeval/metrics')]:
 module=types.ModuleType(name);module.__path__=[str(path)];module.__package__=name;sys.modules[name]=module
HOTA=importlib.import_module(namespace+'.metrics.hota').HOTA
CLEAR=importlib.import_module(namespace+'.metrics.clear').CLEAR
Identity=importlib.import_module(namespace+'.metrics.identity').Identity
METRICS={'clear':CLEAR({'PRINT_CONFIG':False,'THRESHOLD':.5}),'identity':Identity({'PRINT_CONFIG':False,'THRESHOLD':.5}),'hota':HOTA()}
print('Official TrackEval revision verified:',TRACKEVAL_REV)

In [ ]:
def iou_matrix(a,b):
 a=np.asarray(a,float).reshape(-1,4);b=np.asarray(b,float).reshape(-1,4)
 inter=np.maximum(0,np.minimum(a[:,None,2:],b[None,:,2:])-np.maximum(a[:,None,:2],b[None,:,:2])).prod(2)
 union=(a[:,2:]-a[:,:2]).prod(1)[:,None]+(b[:,2:]-b[:,:2]).prod(1)[None,:]-inter
 return np.divide(inter,union,out=np.zeros_like(inter),where=union>0)

def match_boxes(track_boxes,det_boxes,threshold):
 sim=iou_matrix(track_boxes,det_boxes);n,m=sim.shape
 if not n or not m:return []
 # Lexicographic maximum cardinality among allowed pairs, then maximum IoU.
 # A bonus > maximal total IoU makes this objective explicit.
 score=np.zeros((n+m,n+m));score[:n,:m]=np.where(sim>=threshold,n+m+1+sim,0)
 rows,cols=linear_sum_assignment(-score)
 return [(int(i),int(j)) for i,j in zip(rows,cols) if i<n and j<m and sim[i,j]>=threshold]

def make_metric_data(gt_frames,pred_frames):
 assert len(gt_frames)==len(pred_frames)
 gids=sorted({int(i) for f in gt_frames for i in f['ids']});pids=sorted({int(i) for f in pred_frames for i in f['ids']})
 gm={x:i for i,x in enumerate(gids)};pm={x:i for i,x in enumerate(pids)}
 data={'num_gt_ids':len(gids),'num_tracker_ids':len(pids),'num_gt_dets':sum(len(f['ids']) for f in gt_frames),'num_tracker_dets':sum(len(f['ids']) for f in pred_frames),'num_timesteps':len(gt_frames),'gt_ids':[],'tracker_ids':[],'similarity_scores':[]}
 for g,p in zip(gt_frames,pred_frames):
  assert len(set(g['ids']))==len(g['ids']) and len(set(p['ids']))==len(p['ids'])
  data['gt_ids'].append(np.array([gm[int(i)] for i in g['ids']],int));data['tracker_ids'].append(np.array([pm[int(i)] for i in p['ids']],int))
  data['similarity_scores'].append(iou_matrix(g['boxes'],p['boxes']))
 return data

def metric_row(results):
 c,i,h=results['clear'],results['identity'],results['hota']
 return {**{k:float(c[k]) for k in ['MOTA','CLR_TP','CLR_FP','CLR_FN','IDSW']},**{k:float(i[k]) for k in ['IDF1','IDTP','IDFP','IDFN']},**{k:float(np.mean(h[k])) for k in ['HOTA','DetA','AssA']}}

def official(data):return {name:metric.eval_sequence(data) for name,metric in METRICS.items()}

def simple_case(ids,fp_per_frame=0,overlap=1.):
 # One GT per frame. -1 means missing detection. Extra predictions never overlap.
 gt=[];pred=[]
 for t,tid in enumerate(ids):
  gt.append({'ids':[0],'boxes':[[0,0,10,10]]});ps=[];boxes=[]
  if tid>=0:
   dx=10*(1-overlap)/(1+overlap);ps.append(int(tid));boxes.append([dx,0,10+dx,10])
  for j in range(fp_per_frame):ps.append(100+j);boxes.append([30+15*j,0,40+15*j,10])
  pred.append({'ids':ps,'boxes':boxes})
 return make_metric_data(gt,pred)

In [ ]:
import ipywidgets as W
from IPython.display import display
from IPython.utils.capture import capture_output
PANELS={}
def panel(name, fn, controls):
    out=W.Output(); state={'calls':0,'last':None}
    def refresh(change=None):
        with capture_output(display=True) as captured:
            result=fn(**{key:control.value for key,control in controls.items()})
        items=[{'output_type':'display_data','data':item.data,'metadata':item.metadata}
               for item in captured.outputs]
        if captured.stdout:
            items.append({'output_type':'stream','name':'stdout','text':captured.stdout})
        if captured.stderr:
            items.append({'output_type':'stream','name':'stderr','text':captured.stderr})
        out.outputs=tuple(items)
        state['last']=result; state['calls']+=1
    for control in controls.values():
        if hasattr(control,'continuous_update'): control.continuous_update=False
        control.observe(refresh,names='value')
    button=W.Button(description='Повторить опыт',icon='refresh')
    button.on_click(refresh)
    PANELS[name]={'controls':controls,'state':state,'refresh':refresh,'button':button}
    display(W.VBox([W.HBox(list(controls.values()),
                           layout=W.Layout(flex_flow='row wrap')),button,out]))
    refresh()

## 1. Confidence: перекрытие, фон и пороги отбора
High: score ≥ high. Low: low ≤ score < high. Остальные кандидаты игнорируются.
Изменяются не только пороги: перекрытие уменьшает scores двух истинных объектов,
а фон даёт управляемое число ложных кандидатов и их scores. Сдвиг scores меняет
шкалу confidence при тех же рамках. Значения заданы синтетически: это не
обученный детектор и не эксперимент по его калибровке.

GT показывается только для проверки отбора: трекер её не получает. Таблица TP/FP/FN
здесь относится к кандидатам после порога, **до association**; это не MOT-метрики.
Зелёный — high, оранжевый — low, серый — ignored; пунктир рамки обозначает фон.

In [ ]:
def show_confidence(low,high,occlusion=0.,clutter=2,background_score=.25,score_bias=0.):
 if not 0<=low<high<=1:
  print('Нужны 0 ≤ low < high ≤ 1: перекрывающиеся группы недопустимы.');return {'valid':False}
 truth=np.r_[np.ones(4,bool),np.zeros(clutter,bool)]
 base=np.array([.92,.75,.60,.45]);base[2:]-=occlusion
 ratios=np.array([1.,.32,.8,.55,.92,.18,.7,.4])[:clutter]
 scores=np.clip(np.r_[base,background_score*ratios]+score_bias,0,1)
 hi=scores>=high;lo=(scores>=low)&~hi;keep=hi|lo
 colors=np.where(hi,'seagreen',np.where(lo,'darkorange','lightgray'))
 labels=[f'D{j} / '+('GT '+str(j) if j<4 else 'фон') for j in range(len(scores))]
 fig,axs=plt.subplots(1,3,figsize=(14,4.5),constrained_layout=True)
 for j in range(len(scores)):
  x=(j%4)*20;y=(j//4)*20
  axs[0].add_patch(Rectangle((x,y),13,12,fill=False,color=colors[j],lw=2,ls='-' if truth[j] else '--'))
  axs[0].text(x+6.5,y+6,f'D{j}\n{scores[j]:.2f}',ha='center',va='center',fontsize=9)
 axs[0].set(xlim=(-4,78),ylim=(57,-6),aspect='equal',title='Те же рамки; GT только у оценщика');axs[0].axis('off')
 axs[1].barh(labels,scores,color=colors);axs[1].axvline(low,color='darkorange',ls='--',label='low');axs[1].axvline(high,color='seagreen',ls='--',label='high')
 axs[1].set(xlim=(0,1),xlabel='synthetic score',title='Confidence и принадлежность группе');axs[1].invert_yaxis();axs[1].legend(fontsize=8)
 def counts(mask):return {'TP':int((mask&truth).sum()),'FP':int((mask&~truth).sum()),'FN':int((~mask&truth).sum())}
 high_counts=counts(hi);all_counts=counts(keep);keys=['TP','FP','FN'];x=np.arange(3)
 axs[2].bar(x-.18,[high_counts[k] for k in keys],width=.36,label='high only',color='seagreen');axs[2].bar(x+.18,[all_counts[k] for k in keys],width=.36,label='high + low',color='darkorange')
 axs[2].set(xticks=x,xticklabels=keys,ylim=(0,max(4,clutter)+1),title='Отбор кандидатов, до association');axs[2].legend(fontsize=8);axs[2].grid(axis='y',alpha=.2)
 plt.show();plt.close(fig)
 print('High indices:',np.flatnonzero(hi).tolist(),'low:',np.flatnonzero(lo).tolist(),'ignored:',np.flatnonzero(~keep).tolist())
 print('High only:',high_counts,'high+low:',all_counts,'; GT используется только оценщиком отбора.')
 return {'valid':True,'high':int(hi.sum()),'low':int(lo.sum()),'high_counts':high_counts,'kept_counts':all_counts,'scores':scores.tolist(),'groups':np.where(hi,'high',np.where(lo,'low','ignored')).tolist()}
panel('confidence',show_confidence,{
 'low':W.FloatSlider(value=.2,min=0,max=.8,step=.05,description='low',continuous_update=False),
 'high':W.FloatSlider(value=.6,min=.2,max=.95,step=.05,description='high',continuous_update=False),
 'occlusion':W.FloatSlider(value=0,min=0,max=.6,step=.05,description='Ослабление',continuous_update=False,style={'description_width':'initial'}),
 'clutter':W.IntSlider(value=2,min=0,max=8,description='Фон: число',continuous_update=False,style={'description_width':'initial'}),
 'background_score':W.FloatSlider(value=.25,min=0,max=.9,step=.05,description='Score фона',continuous_update=False,style={'description_width':'initial'}),
 'score_bias':W.FloatSlider(value=0,min=-.4,max=.4,step=.05,description='Сдвиг scores',continuous_update=False,style={'description_width':'initial'})})
def ce_confidence_enable(change=None):
 PANELS['confidence']['controls']['background_score'].disabled=PANELS['confidence']['controls']['clutter'].value==0
PANELS['confidence']['controls']['clutter'].observe(ce_confidence_enable,names='value');ce_confidence_enable()

### Проверка на стенде 1

1. Зафиксируйте low=.2, high=.6 и остальные параметры по умолчанию. Поднимите
   только «Ослабление»: какие истинные объекты переходят high → low → ignored?
   Выпишите изменение TP/FN для high-only и high+low.
2. При одинаковом ослаблении увеличьте только score фона. Найдите диапазон,
   где снижение low возвращает истинный объект вместе с фоном. Сравните TP/FP/FN,
   а не только число выбранных рамок.
3. Измените только число фоновых кандидатов. Почему одинаковая доля отобранного
   фона может означать разное абсолютное число FP?
4. При неизменных рамках и GT добавьте сдвиг scores +.2, оставив пороги.
   Какие кандидаты изменили группу? Почему это ещё не изменение геометрии и identity?
5. Проверьте score ровно на low/high, затем low≥high и отсутствие фона.
   Объясните, почему перекрывающиеся диапазоны должны отвергаться явно.

## 2. Две стадии: приоритет, gate и допустимость lost
Сначала high сопоставляются с active/lost tracks. Затем low получают только
оставшиеся ранее active tracks. Low не создаёт рождения. Переключатель lost
делает T1 потерянным ещё до этого шага: его нельзя восстановить через low.
Порог IoU задаёт допустимость до назначения, а не фильтрует пары после него.

«Все вместе» — намеренная абляция приоритета: один assignment использует high
и low одновременно. При этом сохраняется запрет lost→low. «Low births» — другая
абляция, меняющая только рождение из оставшихся low. Полная стоимость — 1−IoU;
helper максимизирует число допустимых пар, затем IoU, а не воспроизводит весь ByteTrack.

In [ ]:
def ce_assign_allowed(sim,allowed):
 n,m=sim.shape
 if not n or not m:return []
 score=np.zeros((n+m,n+m));score[:n,:m]=np.where(allowed,n+m+1+sim,0)
 rr,cc=linear_sum_assignment(-score)
 return [(int(i),int(j)) for i,j in zip(rr,cc) if i<n and j<m and allowed[i,j]]

def show_stages(use_low,low_births,weak_offset,competing_low,gate=.2,priority='high_first',lost_second=False):
 tracks=np.array([[0,0,10,10],[25,0,35,10]],float)
 det=np.array([[1,0,11,10],[25+weak_offset,0,35+weak_offset,10],[60,0,70,10]],float)
 if competing_low:det=np.vstack([det,tracks[0]])
 high=[0];low=list(range(1,len(det)));sim=iou_matrix(tracks,det)
 high_allowed=(sim>=gate)&np.isin(np.arange(len(det)),high)[None,:]
 low_base=(sim>=gate)&np.isin(np.arange(len(det)),low)[None,:]&np.array([True,not lost_second])[:,None]
 def solve(mode):
  first=ce_assign_allowed(sim,high_allowed)
  low_mask=low_base.copy();low_mask[[i for i,j in first]]=False
  if mode=='high_first':pairs=first+(ce_assign_allowed(sim,low_mask) if use_low else [])
  elif mode=='all_together':pairs=ce_assign_allowed(sim,high_allowed|(low_base if use_low else False))
  else:raise ValueError('unknown priority')
  used={j for i,j in pairs};births=[j for j in high if j not in used]
  if low_births:births += [j for j in low if j not in used]
  return [(i,j,'high' if j in high else 'low') for i,j in pairs],births,low_mask
 pairs,births,low_mask=solve(priority);reference,reference_births,_=solve('high_first')
 visible_low=(low_mask if priority=='high_first' else low_base) if use_low else np.zeros_like(low_base)
 fig,axs=plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
 ax=axs[0,0]
 for i,b in enumerate(tracks):
  ax.add_patch(Rectangle((b[0],2),10,5,fill=False,color='navy',lw=2,ls='--' if i==1 and lost_second else '-'));ax.text(b[0]+5,9,f'T{i} '+('lost' if i==1 and lost_second else 'active'),ha='center',fontsize=9)
 for j,b in enumerate(det):
  y=-18 if j==3 else -7
  ax.add_patch(Rectangle((b[0],y),10,5,fill=False,color='seagreen' if j==0 else 'darkorange',lw=2));ax.text(b[0]+5,y-3,f'D{j}'+(' NEW' if j in births else ''),ha='center',fontsize=9)
 for i,j,stage in pairs:
  color='seagreen' if stage=='high' else 'darkorange'
  if j==3:
   # The low duplicate sits below D0. Route through the free left lane so
   # the association cannot look like a T -> D0 -> D3 causal chain.
   start=tracks[i,0]+5
   ax.plot([start,start,-4,-4],[2,-.5,-.5,-15.5],color=color,lw=1.3)
   ax.annotate('',xy=(det[j,0],-15.5),xytext=(-4,-15.5),arrowprops={'arrowstyle':'->','color':color})
  else:ax.annotate('',xy=(det[j,0]+5,-2),xytext=(tracks[i,0]+5,2),arrowprops={'arrowstyle':'->','color':color})
 ax.set(xlim=(-7,78),ylim=(-24 if competing_low else -14,13),yticks=[],xlabel='x',title='Выбранный порядок: '+priority)
 def matrix(ax,a,title,mask=False):
  ax.imshow(a,vmin=0,vmax=1,cmap='YlGnBu',aspect='auto')
  for (i,j),value in np.ndenumerate(a):ax.text(j,i,('да' if value else '×') if mask else f'{value:.2f}',ha='center',va='center',color='white' if value>.65 else 'black')
  ax.set(xticks=range(len(det)),xticklabels=[f'D{j}' for j in range(len(det))],yticks=[0,1],yticklabels=['T0','T1'],title=title)
 matrix(axs[0,1],1-sim,'Стоимость 1−IoU, до gate')
 matrix(axs[1,0],high_allowed,'High: геометрия и группа',True)
 matrix(axs[1,1],visible_low,('Low: active'+(' + строки после high' if priority=='high_first' else ', одновременно с high')) if use_low else 'Low: использование отключено',True)
 plt.show();plt.close(fig)
 print('Selected:',pairs,'births:',births,'; HIGH-FIRST reference:',reference,'births:',reference_births)
 print('IoU matrix:',np.round(sim,3).tolist(),'; gate:',gate,'; T1 previous state:','lost' if lost_second else 'active')
 return {'pairs':pairs,'births':births,'reference_pairs':reference,'reference_births':reference_births,'iou':sim.tolist(),'high_allowed':high_allowed.tolist(),'low_allowed':visible_low.tolist(),'priority':priority}
panel('stages',show_stages,{
 'use_low':W.Checkbox(value=True,description='Вторая стадия'),
 'low_births':W.Checkbox(value=False,description='Рождения из low'),
 'weak_offset':W.IntSlider(value=1,min=0,max=15,description='Сдвиг low',continuous_update=False,style={'description_width':'initial'}),
 'competing_low':W.Checkbox(value=False,description='Low с лучшим IoU'),
 'gate':W.FloatSlider(value=.2,min=0,max=1,step=.05,description='IoU gate',continuous_update=False,style={'description_width':'initial'}),
 'priority':W.Dropdown(options=[('High первым','high_first'),('Все вместе: абляция','all_together')],value='high_first',description='Порядок',style={'description_width':'initial'}),
 'lost_second':W.Checkbox(value=False,description='T1 уже lost')})
def ce_stages_enable(change=None):
 controls=PANELS['stages']['controls']
 controls['priority'].disabled=not controls['use_low'].value
 controls['lost_second'].disabled=not controls['use_low'].value
PANELS['stages']['controls']['use_low'].observe(ce_stages_enable,names='value');ce_stages_enable()

### Проверка на стенде 2

1. При исходных рамках отключите/включите вторую стадию. Выпишите пары в исходных
   индексах и объясните, какую строку получает low. Сравните с матрицей допустимости.
2. Включите «Low с лучшим IoU», оставьте остальные параметры. Сравните «High первым»
   и «Все вместе»: какой D забирает T0 и какой оставшийся high создаёт новый ID?
   Укажите IoU, stage и рождения; число связей само по себе не объясняет разницу.
3. С теми же наблюдениями сделайте T1 lost. Почему его геометрически хорошая low
   остаётся запрещённой? Объясните отличие запрета состояния от IoU gate.
4. При сдвиге low=5 измените только IoU gate .2 → .5. Найдите порог, на котором
   пара T1–D1 исчезает. Почему снижение score не расширяет геометрическую допустимость?
5. Разрешите low births при неизменных парах. Сопоставьте полезное продолжение,
   рождение из удалённого фона D2 и рождение из дубликата D3. Это намеренные нарушения
   штатного правила, а не дополнительные успешные обнаружения людей.

## 3. История ID: дробление, возврат, пропуск и локализация
MOTA считает detection errors и ID switches; IDF1 выбирает одно глобальное
соответствие траекторий; HOTA разделяет detection и association. Вход этого
стенда — одна GT-траектория, история выданных ID и рамки. Выход — официальные
метрики и матрица C совместимых наблюдений GT ↔ predicted ID за весь клип.
Серый вариант справа сохраняет **те же рамки и те же пропуски**, но выдаёт
стабильный ID: разница с ним изолирует identity от detection/localization.

Шесть параметров задают длину клипа, момент первого изменения ID, дальнейший
шаблон ID, длину и начало пропуска, IoU оставшихся рамок. Шаблон действует
после границы split; при split=N он не используется. Положение пропуска имеет
смысл только при его ненулевой длине. Пустые predictions допустимы.

In [ ]:
def metric_control_frames(ids, overlaps, extras=0):
    ids=np.asarray(ids,int);overlaps=np.broadcast_to(np.asarray(overlaps,float),ids.shape)
    if np.any((overlaps<0)|(overlaps>1)):raise ValueError('IoU outside [0,1]')
    gt=[];pred=[]
    for tid,ov in zip(ids,overlaps):
        gt.append({'ids':[0],'boxes':np.array([[0.,0.,10.,10.]])})
        pids=[];boxes=[]
        if tid>=0:
            dx=10*(1-ov)/(1+ov)
            # Synthetic setpoints at a threshold must not land just below it
            # because of the inverse-IoU formula's floating-point roundoff.
            if 0<ov<1:dx=np.nextafter(dx,-np.inf)
            pids.append(int(tid));boxes.append([dx,0,10+dx,10])
        for j in range(extras):pids.append(100+j);boxes.append([30+15*j,0,40+15*j,10])
        pred.append({'ids':pids,'boxes':np.array(boxes,float).reshape(-1,4)})
    return gt,pred

def show_fragment(split,length=8,pattern='single split',gap=0,gap_start=0,overlap=1.):
    length=int(length);split=int(np.clip(split,0,length));gap=int(np.clip(gap,0,length))
    gap_start=int(np.clip(gap_start,0,length-gap));t=np.arange(length)
    if pattern=='single split':ids=np.where(t<split,0,1)
    elif pattern=='return':ids=np.where((t>=split)&(t<split+max(1,(length-split)//2)),1,0)
    elif pattern=='alternating':ids=np.where(t<split,0,(t-split+1)%2)
    else:raise ValueError('unknown identity pattern')
    ids[gap_start:gap_start+gap]=-1
    gt,pred=metric_control_frames(ids,overlap);data=make_metric_data(gt,pred)
    full=official(data);res=metric_row(full)
    reference_ids=np.where(ids<0,-1,0)
    rg,rp=metric_control_frames(reference_ids,overlap);ref=metric_row(official(make_metric_data(rg,rp)))
    actual_ids=sorted({int(i) for f in pred for i in f['ids']})
    C=np.zeros((1,len(actual_ids)),int)
    for gi,pi,sim in zip(data['gt_ids'],data['tracker_ids'],data['similarity_scores']):
        rr,cc=np.nonzero(sim>=.5);np.add.at(C,(gi[rr],pi[cc]),1)
    pairs=[]
    if C.size:
        rows,cols=linear_sum_assignment(-C);pairs=[(int(r),int(actual_ids[c])) for r,c in zip(rows,cols)]
    fig,axs=plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
    ax=axs[0,0];ax.scatter(t,np.ones(length),color='black',s=80,marker='s')
    for tid in sorted(set(ids.tolist())):
        mask=ids==tid
        ax.scatter(t[mask],np.zeros(mask.sum()),color='lightgray' if tid<0 else plt.cm.tab10(tid),s=90,marker='x' if tid<0 else 's',label='missing' if tid<0 else f'P{tid}')
    ax.axvspan(gap_start-.4,gap_start+gap-.6,color='gray',alpha=.12) if gap else None
    ax.axvline(split-.5,color='gray',ls='--',lw=.8)
    ax.set(ylim=(-.6,1.6),xlim=(-.6,length-.4),yticks=[0,1],yticklabels=['predicted ID','GT 0'],xlabel='frame',title=f'{pattern}; N={length}; actual split={split}')
    ax.set_xticks(t[::max(1,length//12)]);ax.legend(loc='upper right',fontsize=8,ncol=3)
    ax=axs[0,1];keys=['MOTA','IDF1','HOTA'];x=np.arange(3)
    ax.bar(x-.19,[ref[k] for k in keys],width=.38,color='gray',label='stable-ID: same boxes/misses')
    ax.bar(x+.19,[res[k] for k in keys],width=.38,color='seagreen',label='selected ID history')
    ax.set(xlim=(-.6,2.6),xticks=x,xticklabels=keys,ylim=(min(0,min(res[k] for k in keys)-.1),1.1),title='Official TrackEval');ax.axhline(0,color='black',lw=.6);ax.legend(fontsize=8)
    ax=axs[1,0]
    if C.size:
        ax.imshow(C,aspect='auto',cmap='Blues',vmin=0,vmax=max(1,length))
        for j,v in enumerate(C[0]):
            selected=(0,actual_ids[j]) in pairs
            ax.text(j,0,str(v)+(' *' if selected else ''),ha='center',va='center',fontsize=14,color='white' if v>.65*length else 'black')
        ax.set(xticks=range(len(actual_ids)),xticklabels=[f'P{i}' for i in actual_ids],yticks=[0],yticklabels=['GT 0'],title='C: compatibility at IoU >= .5; * global ID pair')
    else:ax.text(.5,.5,'C has zero columns: no predictions',ha='center',va='center',transform=ax.transAxes);ax.set_axis_off()
    ax=axs[1,1];ax.axis('off')
    text=[f'GT detections = {length}; predictions = {data["num_tracker_dets"]}',f'IDTP={res["IDTP"]:.0f}; IDFN={res["IDFN"]:.0f}; IDFP={res["IDFP"]:.0f}',f'CLEAR TP={res["CLR_TP"]:.0f}; FN={res["CLR_FN"]:.0f}; FP={res["CLR_FP"]:.0f}',f'IDSW={res["IDSW"]:.0f}; DetA={res["DetA"]:.4f}; AssA={res["AssA"]:.4f}',f'Stable-ID IDF1={ref["IDF1"]:.4f}; selected IDF1={res["IDF1"]:.4f}','C contains compatible detections; HOTA uses its own matching.']
    ax.text(0,.94,'\n\n'.join(text),va='top',fontsize=11)
    plt.show();plt.close(fig)
    print('IDs:',ids.tolist(),'; C:',C.tolist(),'; global ID pairs:',pairs)
    print({k:round(res[k],4) for k in ['MOTA','IDF1','HOTA','IDSW','IDTP','IDFN','IDFP','DetA','AssA']})
    return {**res,'ids':ids.tolist(),'C':C.tolist(),'global_pairs':pairs,'reference':ref,'length':length,'gap_start':gap_start}

fragment_controls={
 'split':W.IntSlider(value=4,min=0,max=8,description='Граница ID',continuous_update=False),
 'length':W.IntSlider(value=8,min=4,max=24,step=4,description='Кадров N',continuous_update=False),
 'pattern':W.Dropdown(options=['single split','return','alternating'],value='single split',description='История ID'),
 'gap':W.IntSlider(value=0,min=0,max=8,description='Пропущено',continuous_update=False),
 'gap_start':W.IntSlider(value=0,min=0,max=8,description='Начало gap',continuous_update=False),
 'overlap':W.FloatSlider(value=1.,min=.1,max=1.,step=.05,description='IoU рамок',continuous_update=False)}
for control in fragment_controls.values():
    if 'description_width' in control.style.traits():control.style.description_width='initial'
    control.layout.width='285px'
panel('fragment',show_fragment,fragment_controls)
def fragment_control_limits(change=None):
    c=fragment_controls;n=c['length'].value
    c['split'].max=n;c['gap'].max=n;c['gap_start'].max=max(0,n-c['gap'].value)
    c['gap_start'].disabled=c['gap'].value==0
    c['pattern'].disabled=c['split'].value==n
for key in ['length','gap','split']:fragment_controls[key].observe(fragment_control_limits,names='value')
fragment_control_limits()

### Проверка на стенде 3

1. Оставьте N=8, single split, без пропуска, IoU=1. Сравните split=1 и4:
   выпишите IDSW, C, глобальную пару, IDTP и IDF1. Почему одинаковое число
   переключений не означает одинаковую длительность ошибки identity?
2. При том же split сравните single split, return и alternating. Найдите
   историю с одинаковой локализацией, но другим числом IDSW и AssA; объясните
   роль возврата старого ID и повторных переключений.
3. Установите ненулевой пропуск, затем измените только его начало: до,
   поперёк и после split. Как меняются IDTP, FN и последний matched ID?
   Сопоставьте те же данные со stable-ID reference.
4. При фиксированной истории ID снизьте IoU через .5. Что происходит с
   совместимостью C и IDTP? Почему потеря identity-совместимости здесь вызвана
   локализацией, а не дополнительными переименованиями?
5. Увеличьте только N при той же границе split, а затем уберите все predictions.
   Выпишите знаменатели и нулевые случаи. Не интерпретируйте отсутствие TP
   как свидетельство хорошей association.

## 4. Ошибки детекции: пропуски, фон, время и identity
GT содержит один объект во всех восьми кадрах. Число и расположение пропусков,
дополнительные рамки, их ID, локализация и правило восстановления ID управляются
независимо. Выбираются готовые предсказания для анализа оценщика: это не онлайн-трекер.
Непересекающиеся дополнительные рамки всегда фон, но присутствующий основной bbox
при низком IoU тоже может не пройти matching. Все метрики считает официальный TrackEval.

Сравнение «без пропусков» сохраняет FP, их ID и локализацию: убирается только
причина missing observations. «ID фона каждый кадр» меняет число ложных траекторий,
сохраняя те же рамки и FP; равенство метрик при разных ID — содержательный результат.
Переключатель сброса ID действует только после внутреннего пропуска, когда истинный
объект уже наблюдался: начальный пропуск не создаёт IDSW из несуществующего прошлого.

In [ ]:
def ce_error_case(misses,extras,missing_pattern,fp_identity,overlap,reset_after_gap):
 T=8
 if missing_pattern=='tail':missing=set(range(T-misses,T))
 elif missing_pattern=='head':missing=set(range(misses))
 elif missing_pattern=='spread':missing=set(np.floor((np.arange(misses)+.5)*T/misses).astype(int).tolist()) if misses else set()
 else:raise ValueError('unknown missing pattern')
 gt=[];pred=[];ids=[];tid=0;seen=False;gap=False;fp_ids=set()
 for t in range(T):
  gt.append({'ids':[0],'boxes':[[0,0,10,10]]});out_ids=[];out_boxes=[]
  if t not in missing:
   if reset_after_gap and seen and gap:tid+=1
   dx=10*(1-overlap)/(1+overlap)
   if 0<overlap<1:dx=np.nextafter(dx,-np.inf) # Keep the requested boundary on its inclusive side.
   out_ids.append(tid);out_boxes.append([dx,0,10+dx,10]);ids.append(tid);seen=True;gap=False
  else:ids.append(-1);gap=seen
  for j in range(extras):
   fid=100+j if fp_identity=='persistent' else 100+t*max(1,extras)+j
   fp_ids.add(fid);out_ids.append(fid);out_boxes.append([30+15*j,0,40+15*j,10])
  pred.append({'ids':out_ids,'boxes':out_boxes})
 return make_metric_data(gt,pred),ids,sorted(missing),len(fp_ids)

def show_errors(misses,extras,missing_pattern='tail',fp_identity='persistent',overlap=1.,reset_after_gap=False):
 data,ids,missing,false_ids=ce_error_case(misses,extras,missing_pattern,fp_identity,overlap,reset_after_gap)
 res=metric_row(official(data));reference=metric_row(official(ce_error_case(0,extras,missing_pattern,fp_identity,overlap,reset_after_gap)[0]))
 fig,axs=plt.subplots(2,2,figsize=(12,6.7),constrained_layout=True)
 ax=axs[0,0];ax.scatter(range(8),np.ones(8),c='black',s=100,marker='s')
 for t,tid in enumerate(ids):
  if tid>=0:ax.scatter(t,0,c=[plt.cm.tab10(tid%10)],s=100,marker='s');ax.text(t,-.3,f'T{tid}',ha='center',fontsize=9)
  else:ax.text(t,0,'×',ha='center',va='center',color='crimson',fontsize=18)
 ax.set(xlim=(-.5,7.5),ylim=(-.6,1.5),yticks=[0,1],yticklabels=['основной bbox','GT'],xlabel='frame',title='Наличие bbox и predicted ID')
 vals=[overlap if t not in missing else 0 for t in range(8)]
 axs[0,1].bar(range(8),vals,color=['gray' if t in missing else 'seagreen' for t in range(8)]);axs[0,1].axhline(.5,color='crimson',ls='--',label='CLEAR/Identity .5');axs[0,1].set(ylim=(0,1.1),xlabel='frame',ylabel='IoU',title='Отсутствие и плохая локализация');axs[0,1].legend(fontsize=8)
 keys=['MOTA','IDF1','HOTA'];x=np.arange(3);ax=axs[1,0]
 ax.bar(x-.18,[reference[k] for k in keys],width=.36,label='те же FP/IoU, misses=0',color='gray');ax.bar(x+.18,[res[k] for k in keys],width=.36,label='выбранный опыт',color='navy')
 ax.set(xticks=x,xticklabels=keys,ylim=(min(0,min(res[k] for k in keys)-.15,min(reference[k] for k in keys)-.15),1.1),title='Официальный TrackEval');ax.axhline(0,color='black',lw=.7);ax.legend(fontsize=8)
 count_keys=['CLR_TP','CLR_FN','CLR_FP','IDSW'];counts=[res[k] for k in count_keys]+[false_ids]
 axs[1,1].bar(['TP','FN','FP','IDSW','ID фона'],counts,color=['seagreen','crimson','darkorange','navy','gray']);axs[1,1].set(title='Счётчики и число ложных траекторий',ylim=(0,max(1,max(counts))+1))
 for i,value in enumerate(counts):axs[1,1].text(i,value+.1,str(int(value)),ha='center',fontsize=9)
 plt.show();plt.close(fig)
 print('Missing frames:',missing,'main IDs:',ids,'unique false trajectory IDs:',false_ids)
 print({k:round(res[k],4) for k in keys+['CLR_TP','CLR_FP','CLR_FN','IDSW','IDTP','IDFP','IDFN','DetA','AssA']})
 print('Without misses, same other settings:',{k:round(reference[k],4) for k in keys})
 return {**res,'false_track_ids':false_ids,'missing_frames':missing,'predicted_ids':ids,'reference':reference}
panel('errors',show_errors,{
 'misses':W.IntSlider(value=2,min=0,max=8,description='Пропущено',continuous_update=False,style={'description_width':'initial'}),
 'extras':W.IntSlider(value=1,min=0,max=3,description='Фон/кадр',continuous_update=False,style={'description_width':'initial'}),
 'missing_pattern':W.Dropdown(options=[('В конце','tail'),('В начале','head'),('Равномерно','spread')],value='tail',description='Пропуски',style={'description_width':'initial'}),
 'fp_identity':W.Dropdown(options=[('Постоянные ID','persistent'),('Новые каждый кадр','per_frame')],value='persistent',description='ID фона',style={'description_width':'initial'}),
 'overlap':W.FloatSlider(value=1,min=.05,max=1,step=.05,description='IoU bbox',continuous_update=False,style={'description_width':'initial'}),
 'reset_after_gap':W.Checkbox(value=False,description='Сброс ID после gap')})
def ce_errors_enable(change=None):
 controls=PANELS['errors']['controls'];controls['fp_identity'].disabled=controls['extras'].value==0
 _,ids,missing,_=ce_error_case(controls['misses'].value,0,controls['missing_pattern'].value,'persistent',1.,False)
 observed=[t for t,tid in enumerate(ids) if tid>=0]
 internal_gap=len(observed)>=2 and any(observed[0]<t<observed[-1] for t in missing)
 controls['reset_after_gap'].disabled=not internal_gap
for key in ['misses','extras','missing_pattern']:PANELS['errors']['controls'][key].observe(ce_errors_enable,names='value')
ce_errors_enable()

### Проверка на стенде 4

1. При misses=0, extras=2 и IoU=1 проверьте отрицательную MOTA. Выпишите число GT,
   FP и формулу; затем измените только extras и объясните изменение знаменателей.
2. При misses=2 сравните пропуски в конце и равномерно, сначала без сброса ID,
   затем со сбросом. Почему одинаковое число FN может сопровождаться разной
   identity? Укажите конкретные кадры и IDSW.
3. При фиксированных рамках сравните постоянные ID фона и новые каждый кадр.
   Сопоставьте число ложных траекторий, FP и все три метрики. Почему увеличение
   числа выданных ID не обязано менять MOT-оценку в этой однозначной сцене?
4. С теми же присутствующими bbox снизьте IoU=1 до .45. Объясните различие между
   отсутствующим bbox и рамкой, которую оценщик отверг по локализации. Сверьте
   CLEAR/Identity при .5 и усреднение HOTA по 19 alpha.
5. Установите misses=8: какие quantities ещё определяются по FP, когда TP нет?
   Почему association score без TP не доказывает хорошую identity?

## 5. HOTA: распределение локализации и пороги alpha
Средняя IoU рамок не заменяет качество траекторий. Управляйте IoU начала,
длиной клипа, профилем изменения IoU и IoU конца/нечётных кадров; отдельно
включайте дробление ID. Uniform игнорирует вторую IoU, поэтому её поле отключено;
drift плавно меняет IoU, alternating чередует две точности. GT и число выдаваемых
рамок не меняются. Здесь нет пропущенных detections: при плохой локализации
оценщик может одновременно получить FP и FN.

Шестой параметр — рассматриваемый alpha — **не меняет общую HOTA**, а позволяет
разобрать её реальные TP/FN/FP и DetA/AssA на выбранном пороге. Официальная HOTA
усредняет 19 HOTA-alpha. Пунктирная кривая — те же ID, но одинаковая IoU каждого
кадра, равная средней исходного профиля: это контроль, а не новый детектор.
Полное matching и счётчики берутся из закреплённого TrackEval. В этой версии
assignment по global alignment×IoU строится один раз на кадр, затем фильтруется
для alpha; online tracker не получает информацию будущего.

In [ ]:
def show_alpha(overlap,fragmented,length=8,profile='uniform',tail_overlap=.2,alpha=.5):
    n=int(length);t=np.arange(n)
    if profile=='uniform':ious=np.full(n,float(overlap))
    elif profile=='drift':ious=np.linspace(overlap,tail_overlap,n)
    elif profile=='alternating':ious=np.where(t%2,tail_overlap,overlap)
    else:raise ValueError('unknown localization profile')
    ids=np.where(t<n//2,0,1) if fragmented else np.zeros(n,int)
    gt,pred=metric_control_frames(ids,ious);full=official(make_metric_data(gt,pred));h=full['hota'];row=metric_row(full)
    rg,rp=metric_control_frames(ids,np.mean(ious));reference=official(make_metric_data(rg,rp))['hota']
    alphas=METRICS['hota'].array_labels;selected=int(np.argmin(np.abs(alphas-alpha)));selected_alpha=float(alphas[selected])
    wrong=float(np.sqrt(np.mean(h['DetA'])*np.mean(h['AssA'])))
    fig,axs=plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
    ax=axs[0,0]
    for tid in sorted(set(ids.tolist())):
        mask=ids==tid;ax.scatter(t[mask],ious[mask],s=60,label=f'P{tid}',color=plt.cm.tab10(tid))
    ax.plot(t,ious,color='gray',lw=.7);ax.axhline(selected_alpha,color='black',ls='--',label=f'alpha={selected_alpha:.2f}')
    ax.set(ylim=(0,1.04),xlabel='frame',ylabel='actual bbox IoU',title=f'{profile}: mean IoU={np.mean(ious):.3f}');ax.legend(fontsize=8);ax.grid(alpha=.2)
    ax=axs[0,1]
    for k in ['HOTA','DetA','AssA']:ax.plot(alphas,h[k],'o-',ms=3,label=k)
    ax.plot(alphas,reference['HOTA'],':',color='black',label='uniform at mean IoU')
    ax.axvline(selected_alpha,color='gray',ls='--');ax.set(xlim=(0,1),ylim=(-.03,1.03),xlabel='localization threshold alpha',title='Official threshold curves');ax.legend(fontsize=8);ax.grid(alpha=.2)
    counts={k:int(h['HOTA_'+k][selected]) for k in ['TP','FN','FP']}
    ax=axs[1,0];ax.bar(list(counts),list(counts.values()),color=['seagreen','orange','tomato'])
    for i,k in enumerate(counts):ax.text(i,counts[k]+.2,str(counts[k]),ha='center')
    ax.set(ylim=(0,max(n+1,max(counts.values())+1)),ylabel='detections',title=f'Official counts at alpha={selected_alpha:.2f}')
    ax=axs[1,1];ax.axis('off')
    lines=[f'At alpha={selected_alpha:.2f}: DetA={h["DetA"][selected]:.4f}; AssA={h["AssA"][selected]:.4f}',f'HOTA_alpha={h["HOTA"][selected]:.4f}',f'Correct mean of 19 HOTA_alpha = {row["HOTA"]:.6f}',f'Wrong sqrt(mean DetA * mean AssA) = {wrong:.6f}',f'Uniform same-mean-IoU HOTA = {np.mean(reference["HOTA"]):.6f}',f'CLEAR / Identity at .5: MOTA={row["MOTA"]:.4f}; IDF1={row["IDF1"]:.4f}']
    ax.text(0,.96,'\n\n'.join(lines),va='top',fontsize=11)
    plt.show();plt.close(fig)
    print('IoU by frame:',np.round(ious,3).tolist(),'; IDs:',ids.tolist())
    print('alpha:',round(selected_alpha,2),'counts:',counts,'mean HOTA:',round(row['HOTA'],6),'wrong aggregation:',round(wrong,6))
    return {**row,'ious':ious.tolist(),'selected_alpha':selected_alpha,'counts':counts,'selected_DetA':float(h['DetA'][selected]),'selected_AssA':float(h['AssA'][selected]),'wrong_aggregation':wrong,'reference_HOTA':float(np.mean(reference['HOTA']))}

alpha_controls={
 'overlap':W.FloatSlider(value=.7,min=.05,max=1.,step=.05,description='IoU начало',continuous_update=False),
 'fragmented':W.Checkbox(value=True,description='Два ID вместо одного'),
 'length':W.IntSlider(value=8,min=4,max=24,step=4,description='Кадров N',continuous_update=False),
 'profile':W.Dropdown(options=['uniform','drift','alternating'],value='uniform',description='Профиль IoU'),
 'tail_overlap':W.FloatSlider(value=.2,min=.05,max=1.,step=.05,description='IoU второй',continuous_update=False),
 'alpha':W.FloatSlider(value=.5,min=.05,max=.95,step=.05,description='Разбор alpha',continuous_update=False)}
for control in alpha_controls.values():
    if 'description_width' in control.style.traits():control.style.description_width='initial'
    control.layout.width='285px'
panel('alpha',show_alpha,alpha_controls)
def alpha_control_limits(change=None):alpha_controls['tail_overlap'].disabled=alpha_controls['profile'].value=='uniform'
alpha_controls['profile'].observe(alpha_control_limits,names='value');alpha_control_limits()

### Проверка на стенде 5

1. При uniform и неизменных ID сравните IoU=.7 и.45, затем измените только
   рассматриваемый alpha. Выпишите TP/FN/FP и объясните, почему общий mean HOTA
   не изменяется от выбора показанного порога.
2. Задайте drift или alternating с разными IoU. Сравните кривую с uniform
   при той же средней IoU. Какие кадры теряют TP на разных alpha и почему
   одного среднего качества локализации недостаточно?
3. При фиксированном профиле включите/выключите fragmentation. Покажите,
   что число GT/predictions сохраняется; сопоставьте DetA и AssA на одном alpha.
   Когда локализация скрывает или усиливает видимую цену дробления ID?
4. Найдите профиль, где mean(sqrt(DetA*AssA)) отличается от
   sqrt(mean(DetA)*mean(AssA)). Выпишите оба результата и объясните ошибку
   перестановки нелинейной операции и усреднения.
5. Увеличьте N при том же профиле; затем выберите alpha выше всех IoU.
   Выпишите граничный случай без TP и объясните, почему FN и FP появляются
   одновременно, хотя ни одна рамка не была удалена генератором.

## 6. BYTE в движении: high → low → lost → восстановление

Перемещайте кадр ползунком или кнопками под стендом. Серый контур — точная синтетическая GT, зелёный — high detection, оранжевый — low, синий пунктир — прогноз, цветной ID — **выданная** траектория. Матрицы показывают IoU до назначения; × означает запрет gate. Входные детекции перемешаны: индекс D не является ID.

[BYTE, Algorithm 1](https://arxiv.org/abs/2110.06864): high сначала сопоставляют с active/lost tracks; затем low получают оставшиеся **active** tracks. Low не порождает ID и не восстанавливает уже lost track. Неиспользованный high может создать новый трек. Потерянный ID хранится ограниченное число кадров. Здесь состояние — центр и постоянная скорость, а не полный Kalman/YOLOX ByteTrack. Граница high принята как `score ≥ .6`; в коде авторов используется строгая `>`. Рождение из оставшегося high здесь немедленное: отдельный порог рождения и подтверждение unconfirmed не воспроизводятся. Непарные прогнозы остаются внутри и не выдаются как детекции. Режим «low births» намеренно нарушает штатное правило для абляции.

In [ ]:
# Independent teaching engine: constant centre velocity, not a production tracker.
def adv_box(cx,cy,w=18.,h=22.):
    return np.array([cx-w/2,cy-h/2,cx+w/2,cy+h/2],float)

def adv_scene(miss=1,low_after_gap=False):
    rng=np.random.default_rng(20261027);frames=[]
    for t in range(16):
        truth=[adv_box(15+5*t,40),adv_box(110-5*t,58)]
        det=[]
        for g,b in enumerate(truth):
            if g==0 and 8<=t<8+miss:continue
            score=.28 if (g==0 and 4<=t<=6) or (g==1 and 10<=t<=11) else .9
            if g==0 and low_after_gap and t==8+miss:score=.28
            det.append({'box':b.copy(),'score':score,'feature':np.eye(2)[g]})
        if t in (4,5,6):det.append({'box':adv_box(125,85),'score':.3,'feature':np.array([.7,.7])})
        order=rng.permutation(len(det));det=[det[i] for i in order]
        frames.append({'gt':truth,'det':det,'bg_delta':np.zeros((8,2))})
    return frames

def adv_assignment(sim,cost,rows,cols,gate):
    rows=list(rows);cols=list(cols)
    if not rows or not cols:return []
    C=np.asarray(cost)[np.ix_(rows,cols)];ok=np.asarray(sim)[np.ix_(rows,cols)]>=gate
    # A dummy column per track expresses refusal; forbidden pairs never compete.
    work=np.full((len(rows),len(cols)+len(rows)),1.01)
    work[:,:len(cols)]=np.where(ok,C,1e6)
    rr,cc=linear_sum_assignment(work)
    return [(rows[i],cols[j]) for i,j in zip(rr,cc) if j<len(cols) and ok[i,j] and C[i,j]<=1.]

def adv_run(frames,use_low=True,low_births=False,max_age=3,compensate=False,
            estimator='median',appearance_weight=0.,use_gate=True,gate=.25):
    tracks=[];next_id=1;history=[];gt=[];pred=[]
    for t,f in enumerate(frames):
        deltas=np.asarray(f['bg_delta'],float).reshape(-1,2)
        measured=(np.median(deltas,axis=0) if estimator=='median' else np.mean(deltas,axis=0)) if len(deltas) else np.zeros(2)
        shift=measured if compensate else np.zeros(2)
        before=[]
        for tr in tracks:
            tr['cam_since_obs']+=shift
            tr['box']=tr['box']+np.tile(tr['v']+shift,2)
            tr['miss']+=1
            before.append({'id':tr['id'],'box':tr['box'].copy(),'miss':tr['miss'],
                           'previous_status':tr['status'],'feature':tr['feature'].copy()})
        boxes=np.array([d['box'] for d in f['det']],float).reshape(-1,4)
        sim=iou_matrix([tr['box'] for tr in tracks],boxes)
        app=np.zeros_like(sim)
        for i,tr in enumerate(tracks):
            for j,d in enumerate(f['det']):
                a=tr['feature'];b=d['feature'];app[i,j]=1-float(a@b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-12)
        costs=(1-appearance_weight)*(1-sim)+appearance_weight*app
        hi=[j for j,d in enumerate(f['det']) if d['score']>=.6]
        lo=[j for j,d in enumerate(f['det']) if .15<=d['score']<.6]
        effective_sim=sim if use_gate else np.ones_like(sim)
        first=adv_assignment(effective_sim,costs,range(len(tracks)),hi,gate)
        used_t={i for i,j in first};used_d={j for i,j in first}
        # BYTE second association is reserved for unmatched active tracks.
        eligible=[i for i,tr in enumerate(tracks) if i not in used_t and tr['status']=='active']
        second=adv_assignment(sim,1-sim,eligible,lo,gate) if use_low else []
        pairs=[(i,j,'high') for i,j in first]+[(i,j,'low') for i,j in second]
        used_t.update(i for i,j in second);used_d.update(j for i,j in second)
        matched=[];reactivated=[];out_ids=[];out_boxes=[]
        for i,j,stage in pairs:
            tr=tracks[i];d=f['det'][j]
            if tr['status']=='lost':reactivated.append(tr['id'])
            dt=t-tr['obs_t'];delta=((d['box'][:2]+d['box'][2:])/2-tr['obs_center']-tr['cam_since_obs'])/dt
            tr['v']=delta;tr['box']=d['box'].copy();tr['obs_center']=(d['box'][:2]+d['box'][2:])/2
            tr['obs_t']=t;tr['cam_since_obs']=np.zeros(2);tr['miss']=0;tr['status']='active'
            tr['feature']=d['feature'].copy()
            matched.append((tr['id'],j,stage));out_ids.append(tr['id']);out_boxes.append(tr['box'].copy())
        for i,tr in enumerate(tracks):
            if i not in used_t:tr['status']='lost'
        births=[]
        for j in hi+(lo if low_births else []):
            if j in used_d:continue
            d=f['det'][j];center=(d['box'][:2]+d['box'][2:])/2
            tr={'id':next_id,'box':d['box'].copy(),'v':np.zeros(2),'miss':0,'status':'active',
                'obs_t':t,'obs_center':center,'cam_since_obs':np.zeros(2),'feature':d['feature'].copy()}
            tracks.append(tr);births.append(next_id);out_ids.append(next_id);out_boxes.append(d['box'].copy());next_id+=1
        deleted=[tr['id'] for tr in tracks if tr['miss']>max_age]
        tracks=[tr for tr in tracks if tr['miss']<=max_age]
        gt.append({'ids':list(range(len(f['gt']))),'boxes':f['gt']})
        pred.append({'ids':out_ids,'boxes':out_boxes})
        history.append({'before':before,'sim':sim,'app':app,'cost':costs,'high':hi,'low':lo,
            'first':[(before[i]['id'],j) for i,j in first],
            'second':[(before[i]['id'],j) for i,j in second],
            'low_eligible':[before[i]['id'] for i in eligible],
            'matches':matched,'births':births,'reactivated':reactivated,'deleted':deleted,
            'internal':[(tr['id'],tr['status'],tr['miss']) for tr in tracks],
            'ids':out_ids,'boxes':out_boxes,'shift':measured.copy()})
    return {'history':history,'gt':gt,'pred':pred,'metrics':metric_row(official(make_metric_data(gt,pred)))}

def adv_matrix(ax,data,title,rows,cols,gate=None):
    a=np.asarray(data,float)
    if not a.size:
        ax.text(.5,.5,'Нет кандидатов',ha='center',va='center',transform=ax.transAxes)
        ax.set_title(title);ax.set_axis_off();return
    im=ax.imshow(a,vmin=0,vmax=1,cmap='YlGnBu',aspect='auto')
    for (i,j),v in np.ndenumerate(a):
        ax.text(j,i,f'{v:.2f}'+(' ×' if gate is not None and v<gate else ''),ha='center',va='center',fontsize=10,color='white' if v>.65 else 'black')
    ax.set(xticks=range(a.shape[1]),xticklabels=cols,yticks=range(a.shape[0]),yticklabels=rows,title=title)
    ax.tick_params(axis='x',rotation=25)

def adv_draw(ax,frames,run,t,title):
    f=frames[t];s=run['history'][t]
    for g,b in enumerate(f['gt']):
        ax.add_patch(Rectangle(b[:2],*(b[2:]-b[:2]),fill=False,color='gray',ls=':',lw=2))
        ax.text(b[0],b[3]+3,f'GT {g}',fontsize=9,color='gray')
    for p in s['before']:
        b=p['box'];ax.add_patch(Rectangle(b[:2],*(b[2:]-b[:2]),fill=False,color='navy',ls='--',alpha=.6))
    for j,d in enumerate(f['det']):
        b=d['box'];color='seagreen' if d['score']>=.6 else 'darkorange'
        ax.add_patch(Rectangle(b[:2],*(b[2:]-b[:2]),fill=False,color=color,lw=1.7))
        ax.text(b[0],b[1]-4,f'D{j}: {d["score"]:.2f}',fontsize=8,color=color)
    for tid,b in zip(s['ids'],s['boxes']):
        color=plt.cm.tab10(tid%10);ax.text((b[0]+b[2])/2,(b[1]+b[3])/2,f'ID {tid}',ha='center',va='center',fontsize=10,color=color,bbox={'facecolor':'white','alpha':.8,'edgecolor':'none'})
        pts=[]
        for old in run['history'][:t+1]:
            if tid in old['ids']:
                q=old['boxes'][old['ids'].index(tid)];pts.append((q[:2]+q[2:])/2)
        if pts:pts=np.asarray(pts);ax.plot(pts[:,0],pts[:,1],'.-',color=color,lw=1,alpha=.65)
    ax.set(xlim=(-5,145),ylim=(105,5),aspect='equal',title=title,xlabel='x (pixels)',ylabel='y (pixels)')
    ax.grid(alpha=.12)

def adv_playback(name,max_frame):
    p=PANELS[name];frame=p['controls']['frame']
    play=W.Play(value=frame.value,min=0,max=max_frame,interval=700,description='▶')
    link=W.link((play,'value'),(frame,'value'))
    prev=W.Button(description='Назад',icon='step-backward');nxt=W.Button(description='Вперёд',icon='step-forward');reset=W.Button(description='С начала',icon='refresh')
    def move(delta):
        play.playing=False;frame.value=max(0,min(max_frame,frame.value+delta))
    prev.on_click(lambda b:move(-1));nxt.on_click(lambda b:move(1));reset.on_click(lambda b:move(-frame.value))
    p['playback']={'play':play,'prev':prev,'next':nxt,'reset':reset,'link':link}
    display(W.HBox([play,prev,nxt,reset]))

def show_byte_movie(frame,use_low,low_births,max_age,miss,low_after_gap):
    frames=adv_scene(miss,low_after_gap)
    baseline=adv_run(frames,use_low=False,max_age=max_age)
    run=adv_run(frames,use_low=use_low,low_births=low_births,max_age=max_age)
    s=run['history'][frame];f=frames[frame]
    fig,axs=plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
    adv_draw(axs[0,0],frames,baseline,frame,'High-only: те же входные детекции')
    adv_draw(axs[0,1],frames,run,frame,'Проверяемый вариант: прогноз и выданные ID')
    rows=[f'T{p["id"]}' for p in s['before']];cols=[f'D{j} {d["score"]:.2f}' for j,d in enumerate(f['det'])]
    hi=s['high'];lo=s['low']
    adv_matrix(axs[1,0],s['sim'][:,hi],'1. High: все active/lost',rows,[cols[j] for j in hi],.25)
    eligible=[i for i,p in enumerate(s['before']) if p['id'] in s['low_eligible']]
    adv_matrix(axs[1,1],s['sim'][np.ix_(eligible,lo)],'2. Low: только оставшиеся active',[rows[i] for i in eligible],[cols[j] for j in lo],.25)
    plt.show();plt.close(fig)
    print('Кадр',frame,'high pairs:',s['first'],'low pairs:',s['second'],'low eligible:',s['low_eligible'])
    print('Births:',s['births'],'reactivated:',s['reactivated'],'deleted:',s['deleted'],'internal (ID,status,miss):',s['internal'])
    keys=['MOTA','IDF1','HOTA','CLR_FN','CLR_FP','IDSW']
    print('Весь клип, официальный TrackEval:',{k:round(run['metrics'][k],4) for k in keys})
    print('High-only:',{k:round(baseline['metrics'][k],4) for k in keys})
    return {'frame':frame,'snapshot':s,'metrics':run['metrics'],'baseline':baseline['metrics']}
panel('byte_movie',show_byte_movie,{
    'frame':W.IntSlider(value=5,min=0,max=15,description='Кадр',continuous_update=False),
    'use_low':W.Checkbox(value=True,description='Вторая стадия'),
    'low_births':W.Checkbox(value=False,description='low births'),
    'max_age':W.IntSlider(value=3,min=0,max=5,description='max_age',continuous_update=False),
    'miss':W.IntSlider(value=1,min=0,max=4,description='Пропуск A',continuous_update=False),
    'low_after_gap':W.Checkbox(value=False,description='low после пропуска')})
adv_playback('byte_movie',15)

### Проверка на стенде 6

1. До изменения спрогнозируйте кадры 4–7: сравните high-only и две стадии на **тех же** detections. Выпишите ID, high/low pairs и CLR_FN; объясните, что именно спасла вторая стадия.
2. На кадре 5 включите low births. Какой D породил ложный ID? Проследите его жизнь и сравните FP полного клипа, оставив остальные параметры.
3. Пропуск A начинается с кадра 8. Сравните длину 1 и 3 при `max_age=1`, затем `max_age=4`: найдите последний сохранённый, удалённый и восстановленный ID. Внутренний lost не равен видимой рамке.
4. Включите «low сразу после пропуска», остановитесь на кадре `8+длина пропуска`. Почему геометрически хорошая low не реактивирует lost? Сравните следующий high кадр и правило low births. Запишите eligible tracks второй стадии и конкретный ID.

## 7. Движение камеры: согласовать системы координат до association

Если камера резко сместилась, прогноз в старой системе координат перестаёт пересекать даже правильную детекцию. [BoT-SORT](https://arxiv.org/abs/2206.14651) включает компенсацию движения камеры вместе с motion и appearance. Здесь выделен один механизм: **чистый перенос** по соответствиям фоновых точек. Оценка берётся из наблюдаемых разностей координат; истинный перенос не передаётся трекеру. Медиана — учебная робастная оценка, а не полный affine GMC BoT-SORT. Правильная компенсация переносит прогноз, затем считается IoU; знак переноса важен.

Оба варианта получают одни и те же рамки и фоновые соответствия. Фоновые точки содержат выбросы. Сам график показывает межкадровые векторы смещения, а не оптический поток реального видео. Все метрики относятся к этой синтетике.

In [ ]:
def adv_camera_scene(jump=24,outliers=3):
    rng=np.random.default_rng(71);frames=[];camera=np.zeros(2)
    for t in range(12):
        delta=np.array([float(jump),-4.]) if t==5 else np.zeros(2)
        if t!=5:delta=np.zeros(2)
        camera+=delta
        gt=[adv_box(22+3*t+camera[0],35+camera[1]),adv_box(102-2*t+camera[0],70+camera[1])]
        background=delta+rng.normal(0,.12,(10,2))
        if t==5 and outliers:
            background[:outliers]=delta+np.array([-35.,20.])+rng.normal(0,.12,(outliers,2))
        det=[{'box':b.copy(),'score':.9,'feature':np.eye(2)[g]} for g,b in enumerate(gt)]
        if t%2:det.reverse()
        frames.append({'gt':gt,'det':det,'bg_delta':background,'true_delta':delta})
    return frames

def show_camera_motion(frame,jump,outliers,estimator,compensate):
    frames=adv_camera_scene(jump,outliers)
    raw=adv_run(frames,max_age=3)
    run=adv_run(frames,max_age=3,compensate=compensate,estimator=estimator)
    s=run['history'][frame];r=raw['history'][frame]
    fig,axs=plt.subplots(2,2,figsize=(12,7),constrained_layout=True)
    adv_draw(axs[0,0],frames,raw,frame,'Без компенсации: прогноз в старой системе')
    adv_draw(axs[0,1],frames,run,frame,'Компенсация' if compensate else 'Переключатель компенсации выключен')
    d=frames[frame]['bg_delta'];ax=axs[1,0]
    ax.scatter(d[:,0],d[:,1],c=np.arange(len(d)),cmap='tab10',s=55,label='observed background deltas')
    ax.scatter(*s['shift'],marker='x',s=150,color='black',label=estimator+' estimate')
    ax.scatter(*frames[frame]['true_delta'],marker='+',s=150,color='red',label='GT: только оценщик')
    ax.set(xlabel='dx',ylabel='dy',title='Перенос из соответствий фоновых точек');ax.grid(alpha=.3);ax.legend(fontsize=8)
    adv_matrix(axs[1,1],s['sim'],'IoU после выбранного преобразования',[f'T{p["id"]}' for p in s['before']],[f'D{j}' for j in range(len(frames[frame]['det']))],.25)
    plt.show();plt.close(fig)
    err=float(np.linalg.norm(s['shift']-frames[frame]['true_delta']))
    print('frame',frame,'estimated delta:',np.round(s['shift'],3),'error (evaluator only):',round(err,3),'pixels')
    print('Pairs:',s['first'],'births:',s['births'],'internal:',s['internal'])
    print('Весь клип, HOTA / IDF1 / IDSW raw:',[round(raw['metrics'][k],4) for k in ['HOTA','IDF1','IDSW']])
    print('Выбранный режим:',[round(run['metrics'][k],4) for k in ['HOTA','IDF1','IDSW']])
    return {'frame':frame,'snapshot':s,'error':err,'metrics':run['metrics'],'raw':raw['metrics']}
panel('camera_motion',show_camera_motion,{
    'frame':W.IntSlider(value=5,min=0,max=11,description='Кадр',continuous_update=False),
    'jump':W.IntSlider(value=24,min=-24,max=28,step=4,description='Скачок x',continuous_update=False),
    'outliers':W.IntSlider(value=3,min=0,max=7,description='Выбросы /10',continuous_update=False),
    'estimator':W.Dropdown(options=['median','mean'],value='median',description='Оценка'),
    'compensate':W.Checkbox(value=True,description='Компенсация')})
adv_playback('camera_motion',11)

### Проверка на стенде 7

1. Зафиксируйте кадр 5 и скачок камеры 24. Сравните без компенсации и с ней: выпишите оценённый перенос, IoU правильной пары, births и IDSW. Почему увеличение `max_age` само не исправляет систему координат?
2. С теми же точками и числом выбросов сравните mean и median. Запишите ошибку переноса и matrix gate. Затем увеличьте долю выбросов до большинства: найдите предел устойчивости медианы.
3. Поставьте скачок 0, затем −24: проверьте знак оценки и направление скорректированной рамки. Объясните, почему вычитание переноса вместо сложения создало бы двойную ошибку.
4. Измените только переключатель компенсации на кадрах 4–6. Проследите переходы ID и отметьте, на каком шаге возникло новое рождение. Приведите пример реального движения, для которого модели чистого переноса недостаточно.

## 8. Внешность и геометрический запрет: полезный признак может ошибаться

Кадры 4–7 образуют трудный участок: движение людей меняется, а рамки сближаются. Здесь нормированные двухмерные признаки заданы **синтетически** и не являются выходом обученной re-ID сети. Варианты distinct / identical / swapped меняют только признаки, сохраняя все рамки. Стоимость — взвешенная сумма `1−IoU` и cosine distance. Геометрический gate применяется **до** назначения; даже идеальное сходство не делает невозможный перенос допустимым. Это управляемая авторская модель взаимодействия сигналов, а не точная формула fusion BoT-SORT.

Слева одинаковый motion-only baseline, справа выбранная комбинация. Матрицы показывают геометрию, внешность и итог; перечёркнутая по gate пара не участвует. Отключённый gate — намеренная абляция. [BoT-SORT, motion/appearance/CMC](https://arxiv.org/abs/2206.14651), [Deep SORT](https://arxiv.org/abs/1703.07402).

In [ ]:
def adv_appearance_scene(mode='distinct'):
    if mode not in ('distinct','identical','swapped'):raise ValueError('feature mode')
    frames=[]
    # Each person approaches, turns, then separates. Motion extrapolation at the
    # turn prefers continuing through; appearance can preserve identity.
    xa=[15,22,29,36,43,51,44,37,30,23,16,9]
    xb=[95,88,81,74,67,59,66,73,80,87,94,101]
    for t in range(12):
        gt=[adv_box(xa[t],48,w=28),adv_box(xb[t],48,w=28)]
        det=[]
        for g,b in enumerate(gt):
            feature=np.eye(2)[g]
            if mode=='identical':feature=np.ones(2)/np.sqrt(2)
            if mode=='swapped' and t>=8:feature=np.eye(2)[1-g]
            det.append({'box':b.copy(),'score':.9,'feature':feature})
        if t%2:det.reverse()
        frames.append({'gt':gt,'det':det,'bg_delta':np.zeros((6,2))})
    return frames

def show_appearance(frame,mode,weight,use_gate,gate):
    frames=adv_appearance_scene(mode)
    raw=adv_run(frames,appearance_weight=0.,gate=gate)
    run=adv_run(frames,appearance_weight=weight,use_gate=use_gate,gate=gate)
    s=run['history'][frame]
    fig,axs=plt.subplots(2,3,figsize=(14,7),constrained_layout=True)
    adv_draw(axs[0,0],frames,raw,frame,'Motion-only: одинаковые boxes')
    adv_draw(axs[0,1],frames,run,frame,'Motion + synthetic appearance')
    ax=axs[0,2]
    ax.bar(['motion HOTA','selected HOTA','motion IDF1','selected IDF1'],[raw['metrics']['HOTA'],run['metrics']['HOTA'],raw['metrics']['IDF1'],run['metrics']['IDF1']],color=['gray','seagreen','gray','navy'])
    ax.set(ylim=(0,1.05),title='Официальные метрики всего клипа');ax.tick_params(axis='x',rotation=28)
    rows=[f'T{p["id"]}' for p in s['before']];cols=[f'D{j}' for j in range(len(frames[frame]['det']))]
    adv_matrix(axs[1,0],s['sim'],'IoU: × запрещено' if use_gate else 'IoU: gate выключен',rows,cols,gate if use_gate else None)
    adv_matrix(axs[1,1],s['app'],'Cosine distance: меньше лучше',rows,cols)
    adv_matrix(axs[1,2],s['cost'],'Fusion cost: меньше лучше',rows,cols)
    plt.show();plt.close(fig)
    print('Frame',frame,'mode',mode,'weight',weight,'gate',gate,'enabled',use_gate)
    print('Pairs:',s['first'],'births:',s['births'],'internal:',s['internal'])
    print('IDSW motion / selected:',raw['metrics']['IDSW'],run['metrics']['IDSW'])
    return {'frame':frame,'snapshot':s,'metrics':run['metrics'],'raw':raw['metrics']}
panel('appearance_gate',show_appearance,{
    'frame':W.IntSlider(value=6,min=0,max=11,description='Кадр',continuous_update=False),
    'mode':W.Dropdown(options=['distinct','identical','swapped'],value='distinct',description='Признак'),
    'weight':W.FloatSlider(value=.8,min=0,max=1,step=.1,description='Вес',continuous_update=False),
    'use_gate':W.Checkbox(value=True,description='Geometry gate'),
    'gate':W.FloatSlider(value=.1,min=.1,max=.6,step=.1,description='IoU gate',continuous_update=False)})
adv_playback('appearance_gate',11)

### Проверка на стенде 8

1. При distinct и весе .8 сравните кадры 4–7 с motion-only. Запишите матрицы, пары и IDSW. Затем при **тех же boxes** включите identical: почему полезная информация исчезает, хотя вычисление cosine всё ещё выполняется?
2. Выберите swapped, кадр 8, вес 1. Сравните gate включён/выключен: выпишите хотя бы одну привлекательную по внешности пару, её IoU и допустимость. Почему геометрический запрет предотвращает ложную связь, хотя признак предпочитает другую пару?
3. При swapped выключите gate и сравните вес 0 и .8, оставив остальные настройки. Проследите отдельно поворот на кадре 6 и порчу признака на кадре 8: где изменился ID из-за движения, а где из-за внешности? Сравните HOTA всего клипа. Почему одинаковое число IDSW не означает одинаковую длительность ошибки и почему увеличение веса не гарантирует улучшение каждого шага?
4. В режиме identical сравните gate .1 и .5 на кадрах после поворота. Какие истинные пары были отвергнуты? Укажите, какую ошибку нельзя исправить качественной внешностью, пока gate запрещает пару.

## 9. Сравнение оценщиков: длина видео, FP/FN и глобальное соответствие ID

Хорошее короткое видео и трудное длинное дают разный вес в общей оценке. Официальный TrackEval объединяет **достаточные статистики** последовательностей своими `combine_sequences`. Простое среднее процентов двух видео означает другой эксперимент. MOTA объединяется через суммарные ошибки и число GT; IDF1 — через IDTP/IDFP/IDFN; HOTA имеет свои веса по TP и alpha. ID не сопоставляются между независимыми видео.

В длинной последовательности GT неизменна, bbox идеальны, но predicted ID дробится на куски. В короткой ID устойчив, могут появляться FP. Матрица C показывает число совместимых наблюдений каждой пары GT ↔ predicted trajectory **за всю длинную sequence**. Глобальное ID-сопоставление выбирает не все ненулевые элементы, а одно взаимно однозначное назначение. Здесь matching однозначен, поэтому эффект изолирован от crowding. Результаты получены официальной реализацией, а не усреднением готовых HOTA.

In [ ]:
def show_pooling(short_length,long_length,fragments,short_fp):
    # Independent sequences; identity labels may repeat across videos.
    short_ids=[0]*short_length
    long_ids=[min(fragments-1,t*fragments//long_length) for t in range(long_length)]
    a=official(simple_case(short_ids,fp_per_frame=short_fp));b=official(simple_case(long_ids))
    combined={name:metric.combine_sequences({'short':a[name],'long':b[name]}) for name,metric in METRICS.items()}
    ra=metric_row(a);rb=metric_row(b);rp=metric_row(combined)
    C=np.array([[long_ids.count(j) for j in range(fragments)]],float)
    ri,cj=linear_sum_assignment(-C);global_pairs=list(zip(ri.tolist(),cj.tolist()))
    keys=['MOTA','IDF1','HOTA'];macro={k:(ra[k]+rb[k])/2 for k in keys}
    fig,axs=plt.subplots(2,2,figsize=(12,6),constrained_layout=True)
    for ax,ids,label in [(axs[0,0],short_ids,'Короткое: стабильный ID'),(axs[0,1],long_ids,'Длинное: дробление ID')]:
        ax.scatter(range(len(ids)),np.zeros(len(ids)),c=ids,cmap='tab10',s=40,marker='s',vmin=0,vmax=9)
        ax.set(ylim=(-.5,.5),yticks=[],xlabel='frame',title=label)
        starts=[0]+[t for t in range(1,len(ids)) if ids[t]!=ids[t-1]]
        for t in starts:ax.text(t,.18,f'ID {ids[t]}',fontsize=9)
    adv_matrix(axs[1,0],C/long_length,'C / N_GT: совместимые траектории',['GT 0'],[f'P{j}' for j in range(fragments)])
    x=np.arange(3);ax=axs[1,1]
    ax.bar(x-.18,[macro[k] for k in keys],width=.36,label='macro mean')
    ax.bar(x+.18,[rp[k] for k in keys],width=.36,label='official pooled')
    ax.set(xticks=x,xticklabels=keys,ylim=(min(0,macro['MOTA']-.1,rp['MOTA']-.1),1.05),title='Два разных способа объединить видео');ax.legend()
    plt.show();plt.close(fig)
    print('Длинное C =',C.astype(int).tolist(),'global ID pairs =',global_pairs,'IDTP =',rb['IDTP'])
    print('GT short/long =',short_length,long_length,'; IDFP/IDFN long =',rb['IDFP'],rb['IDFN'])
    print('Short:',{k:round(ra[k],4) for k in keys},'long:',{k:round(rb[k],4) for k in keys})
    print('Macro:',{k:round(macro[k],4) for k in keys},'pooled:',{k:round(rp[k],4) for k in keys})
    return {'pooled':rp,'macro':macro,'C':C,'global_pairs':global_pairs,'short':ra,'long':rb}
panel('pooling',show_pooling,{
    'short_length':W.IntSlider(value=8,min=4,max=20,step=4,description='Короткое N',continuous_update=False),
    'long_length':W.IntSlider(value=40,min=8,max=80,step=8,description='Длинное N',continuous_update=False),
    'fragments':W.IntSlider(value=4,min=1,max=8,description='Части ID',continuous_update=False),
    'short_fp':W.IntSlider(value=0,min=0,max=2,description='FP/кадр short',continuous_update=False)})

### Проверка на стенде 9

1. Увеличьте длину трудного видео при фиксированных остальных параметрах. До изменения спрогнозируйте направление pooled и macro среднего; запишите оба значения, GT counts и IDTP/IDFN. Почему равный вес видеороликов отвечает другому вопросу?
2. Сравните 1 и 4 части ID при той же длине: выпишите C, глобальные пары и AssA. Почему покадровая локализация остаётся идеальной, а identity падает?
3. Добавьте FP только в короткое видео: сопоставьте изменение MOTA, IDF1, HOTA и их знаменателей. Найдите пример, где среднее отдельных процентов маскирует ошибку длинной последовательности.
4. Измените длину короткого видео, сохранив его качество. Объясните, почему это меняет pooled результат без изменения качества трекера на любом отдельном кадре. Не распространяйте ID одного ролика на другой.

## Дальше
Самостоятельная работа добавляет девять TODO, полный шаг трекера, перенос камеры по фону и восстановимый журнал матриц; затем — val/test, три stress и траектории на реальном видео. Учебные агрегаторы метрик не заменяют matching полного оценщика.

Источники: [ByteTrack](https://arxiv.org/abs/2110.06864), [BoT-SORT](https://arxiv.org/abs/2206.14651), [HOTA](https://arxiv.org/abs/2009.07736), [TrackEval](https://github.com/JonathonLuiten/TrackEval/tree/12c8791b303e0a0b50f753af204249e622d0281a).